In [1]:
import dask.dataframe as dd
import pandas as pd

c:\Users\USER\Documents\openeais\.venv\Lib\site-packages\dask\dataframe\__init__.py:31: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)


In [2]:
import re


def get_numpy_type(dtype_openeais: str):
    split_characters = r"[(,)]"

    substrings = re.split(split_characters, dtype_openeais)

    if substrings[0] in ["VARCHAR", "CHAR"]:
        return "string"
    elif substrings[0] in ["NUMERIC"]:
        if "," in dtype_openeais:
            return "float64"
        else:
            return "Int64"
    else:
        raise

In [3]:
txt_path = "data/국토교통부_건축물대장_표제부(2015년+12월)/MART_DJY_TITLE.txt"
schema_path = "data/schema/schema_건축물대장_표제부(2015년+12월).csv"

results_dir = "data/건축물대장_표제부_2015년_12월"
index = 0

In [4]:
df_schema = pd.read_csv(schema_path, header=None)
df_schema[1] = df_schema[1].apply(get_numpy_type)
df_schema = df_schema.set_index(0)
schema_dict = df_schema[1].to_dict()

In [5]:
schema_dict

{'관리_건축물대장_PK': 'string',
 '대장_구분_코드': 'string',
 '대장_구분_코드_명': 'string',
 '대장_종류_코드': 'string',
 '대장_종류_코드_명': 'string',
 '대지_위치': 'string',
 '도로명_대지_위치': 'string',
 '건물_명': 'string',
 '시군구_코드': 'string',
 '법정동_코드': 'string',
 '대지_구분_코드': 'string',
 '번': 'string',
 '지': 'string',
 '특수지_명': 'string',
 '블록': 'string',
 '로트': 'string',
 '외필지_수': 'Int64',
 '알수없음1': 'string',
 '새주소_도로_코드': 'string',
 '새주소_법정동_코드': 'string',
 '새주소_지상지하_코드': 'string',
 '새주소_본_번': 'Int64',
 '새주소_부_번': 'Int64',
 '동_명': 'string',
 '주_부속_구분_코드': 'string',
 '주_부속_구분_코드_명': 'string',
 '알수없음2': 'string',
 '대지_면적(㎡)': 'float64',
 '건축_면적(㎡)': 'float64',
 '건폐_율(%)': 'float64',
 '연면적(㎡)': 'float64',
 '용적_률_산정_연면적(㎡)': 'float64',
 '용적_률(%)': 'float64',
 '구조_코드': 'string',
 '구조_코드_명': 'string',
 '기타_구조': 'string',
 '주_용도_코드': 'string',
 '주_용도_코드_명': 'string',
 '기타_용도': 'string',
 '지붕_코드': 'string',
 '지붕_코드_명': 'string',
 '기타_지붕': 'string',
 '세대_수(세대)': 'Int64',
 '가구_수(가구)': 'Int64',
 '높이(m)': 'float64',
 '지상_층_수': 'Int64

In [6]:
import dask.dataframe as dd
import pandas as pd

ddf: pd.DataFrame = dd.read_csv(
    txt_path,
    encoding="cp949",
    header=None,
    sep="|",
    names=schema_dict.keys(),
    dtype=schema_dict,
    # nrows=1,
)

In [7]:
from dask.diagnostics import ProgressBar

with ProgressBar():
    if index is not None:
        if isinstance(index, int):
            index = ddf.columns[index]

        print("replacing NA...")
        if schema_dict[index] in ["string", str]:
            replace_value = ""
        elif schema_dict[index] in ["Int64", "int64", int]:
            replace_value = 0
        elif schema_dict[index] in ["float64", float]:
            replace_value = 0.0
        else:
            raise NotImplementedError(ddf[index].dtype)
        ddf = ddf.fillna({index: replace_value})

        print("Setting index...")
        ddf = ddf.set_index(index)
    print("Saving...")
    ddf.to_parquet(results_dir)


replacing NA...
Setting index...
[########################################] | 100% Completed | 369.22 s
Saving...
[########################################] | 100% Completed | 283.16 s
